In [ ]:
# Cell 1 - run information (required)
import os, subprocess, sys, random
import numpy as np, torch
ON_COLAB = os.path.isdir("/content")
if ON_COLAB:                       # upload repo.bundle (git bundle create repo.bundle zoheb) when asked
    os.chdir("/content")
    if not os.path.isdir("data266-lab2-team32"):
        from google.colab import files
        files.upload()
        subprocess.run(["git", "clone", "-q", "-b", "zoheb", "repo.bundle", "data266-lab2-team32"], check=True)
    os.chdir("data266-lab2-team32")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

# Part 2 - Fine-tuning Stable Diffusion 1.5 on Naruto images (Zoheb's full pipeline)
DATA 266 Lab 2, Pair_Programming_Team_32. Shreya leads Part 2 (`notebooks/p2_finetune.ipynb`, code in
`part2_diffusion/src/`). The brief requires the partner who does not lead to implement and run the complete pipeline;
this notebook is Zoheb's own implementation of all of 2.1-2.4, written separately. It shares only the 20 fixed
prompts (`part2_diffusion/eval/prompts.csv`, committed before training) and the team's settings (seed 266, rank 4 vs
rank 32, 1500 steps, batch 4, lr 1e-4, DPM-Solver++ 30 steps, guidance 7.5).

Runs on one Colab T4. Everything is saved to Google Drive (`MyDrive/data266_p2_zoheb`) as it is produced; after a
disconnect, **Run all** again and finished training runs and images are reused.

In [ ]:
# packages: diffusers/peft/lpips/torch-fidelity are not on Colab; torchao breaks peft's LoRA loading on Colab
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.42.1", "peft==0.21.2",
                    "torchmetrics==1.9.0", "torch-fidelity==0.3.0", "lpips==0.1.4"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"])
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/data266_p2_zoheb"
else:
    OUT = "part2_diffusion/outputs/big_zoheb"
os.makedirs(OUT, exist_ok=True)
from importlib.metadata import version
print({p: version(p) for p in ["torch", "diffusers", "transformers", "peft", "datasets", "torchmetrics", "lpips"]})

BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DATASET = "lambdalabs/naruto-blip-captions"
CLIP = "openai/clip-vit-base-patch32"
RES, STEPS, BATCH, LR = 512, 1500, 4, 1e-4
RUNS = {"run1_lora_r4": 4, "run2_lora_r32": 32}
MODELS = ["base", *RUNS]
dev = "cuda"

## 2.1 Data preparation
Remove images that cannot be decoded, exact duplicates (MD5 of the file bytes) and near duplicates (64-bit dHash
within 4 bits of a kept image). Split 90/10 with seed 266; test images are used only for evaluation. Resize to 512.

In [ ]:
import hashlib, io, json
from collections import Counter
from datasets import load_dataset, Image as HFImage
from PIL import Image

def dhash(img):
    g = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (g[:, 1:] > g[:, :-1]).flatten()), 2)

raw = load_dataset(DATASET, split="train").cast_column("image", HFImage(decode=False))
rows, md5s, hashes, sizes = [], set(), [], Counter()
removed = {"corrupted": 0, "exact_duplicate": 0, "near_duplicate": 0}
for r in raw:
    b = r["image"]["bytes"]
    try:
        img = Image.open(io.BytesIO(b)).convert("RGB")
    except Exception:
        removed["corrupted"] += 1; continue
    m = hashlib.md5(b).hexdigest()
    if m in md5s:
        removed["exact_duplicate"] += 1; continue
    h = dhash(img)
    if any(bin(h ^ k).count("1") <= 4 for k in hashes):     # ponytail: O(n^2) scan, fine for 1.2k images
        removed["near_duplicate"] += 1; continue
    md5s.add(m); hashes.append(h); sizes[f"{img.width}x{img.height}"] += 1
    rows.append((img.resize((RES, RES), Image.BICUBIC), r["text"]))

order = np.random.default_rng(SEED).permutation(len(rows))
n_test = round(0.1 * len(rows))
test, train = [rows[i] for i in order[:n_test]], [rows[i] for i in order[n_test:]]
words = Counter(w for _, c in rows for w in c.lower().replace(",", " ").split())
stop = {"a", "an", "the", "with", "and", "of", "in", "on", "to", "is", "his", "her", "at", "that", "has", "holding"}
prep = {"loaded": len(raw), "removed": removed, "kept": len(rows), "train": len(train), "test": len(test),
        "original_sizes_top5": dict(sizes.most_common(5)),
        "caption_words_mean": round(float(np.mean([len(c.split()) for _, c in rows])), 1),
        "top_caption_words": dict([(w, n) for w, n in words.most_common(40) if w not in stop][:15])}
print(json.dumps(prep, indent=1))
print("examples:", [c for _, c in train[:4]])

**Dataset.** 1,221 frames from the Naruto anime with BLIP captions (`lambdalabs/naruto-blip-captions`, license as
stated on its Hugging Face card; the characters are copyrighted, so this model is for non-commercial research only).
Two limitations, from the counts above: (1) the captions are short and generic (about 9 words, e.g. "a man with ...")
and never name characters, so the model can learn the drawing style but not who is who; (2) the subjects are skewed
towards men ("man" is far more common than "woman" in the captions) and towards one visual style and palette, so
out-of-domain and female subjects are under-represented.

## 2.2 Fine-tuning: two runs with the same data and step budget
Run 1 = LoRA rank 4, run 2 = LoRA rank 32 (alpha = rank), on the UNet attention projections. The text encoder and VAE
are frozen in fp16; noise-prediction MSE with the DDPM scheduler, fp16 autocast, gradient checkpointing. Both runs see
exactly the same batches (same seed) for 1500 steps.

In [ ]:
import json, time
import torch.nn.functional as F
from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel
from diffusers.utils import convert_state_dict_to_diffusers
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from transformers import CLIPTextModel, CLIPTokenizer

def to_tensor(img):
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 127.5 - 1).permute(2, 0, 1)

def train_lora(run, rank):
    folder = f"{OUT}/loras/{run}"
    if os.path.exists(f"{folder}/stats.json"):
        print(run, "already trained"); return json.load(open(f"{folder}/stats.json"))
    torch.manual_seed(SEED)
    tok = CLIPTokenizer.from_pretrained(BASE, subfolder="tokenizer")
    text_enc = CLIPTextModel.from_pretrained(BASE, subfolder="text_encoder", torch_dtype=torch.float16).to(dev)
    vae = AutoencoderKL.from_pretrained(BASE, subfolder="vae", torch_dtype=torch.float16).to(dev)
    unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet").to(dev)
    sched = DDPMScheduler.from_pretrained(BASE, subfolder="scheduler")
    for m in (text_enc, vae, unet):
        m.requires_grad_(False)
    unet.add_adapter(LoraConfig(r=rank, lora_alpha=rank, init_lora_weights="gaussian",
                                target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
    unet.enable_gradient_checkpointing(); unet.train()
    params = [p for p in unet.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR)
    scaler = torch.amp.GradScaler("cuda")
    gen = torch.Generator().manual_seed(SEED)                    # same batches and noise for both runs
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0, losses = time.time(), []
    for step in range(STEPS):
        idx = torch.randint(len(train), (BATCH,), generator=gen).tolist()
        pix = torch.stack([to_tensor(train[i][0]) for i in idx]).to(dev, torch.float16)
        ids = tok([train[i][1] for i in idx], padding="max_length", max_length=tok.model_max_length,
                  truncation=True, return_tensors="pt").input_ids.to(dev)
        with torch.no_grad():
            latents = vae.encode(pix).latent_dist.sample() * vae.config.scaling_factor
            cond = text_enc(ids)[0]
        noise = torch.randn(latents.shape, generator=gen).to(dev, latents.dtype)
        t = torch.randint(0, sched.config.num_train_timesteps, (BATCH,), generator=gen).to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            pred = unet(sched.add_noise(latents, noise, t), t, cond).sample
        loss = F.mse_loss(pred.float(), noise.float())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        losses.append(loss.item())
        if step % 250 == 0 or step == STEPS - 1:
            print(f"{run} step {step:4d}  loss(avg 100) {np.mean(losses[-100:]):.4f}")
    torch.cuda.synchronize()
    minutes = (time.time() - t0) / 60
    StableDiffusionPipeline.save_lora_weights(folder, unet_lora_layers=convert_state_dict_to_diffusers(
        get_peft_model_state_dict(unet)), safe_serialization=True)
    stats = {"rank": rank, "trainable_params": sum(p.numel() for p in params), "learning_rate": LR, "steps": STEPS,
             "batch_size": BATCH, "peak_gpu_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
             "train_minutes": round(minutes, 1), "gpu_hours": round(minutes / 60, 3),
             "gpu": torch.cuda.get_device_name(0), "loss_curve": [round(float(np.mean(losses[i:i + 50])), 4)
                                                                   for i in range(0, STEPS, 50)]}
    json.dump(stats, open(f"{folder}/stats.json", "w"))
    del unet, vae, text_enc, opt; torch.cuda.empty_cache()
    return stats

stats = {run: train_lora(run, rank) for run, rank in RUNS.items()}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
train_table = pd.DataFrame({r: {k: v for k, v in s.items() if k != "loss_curve"} for r, s in stats.items()}).T
print(train_table.to_string())
for r, s in stats.items():
    plt.plot(range(0, STEPS, 50), s["loss_curve"], label=r)
plt.xlabel("step"); plt.ylabel("training loss (mean of 50 steps)"); plt.legend(); plt.show()

## 2.3 Image generation
All 20 fixed prompts with seeds 0 and 1 for the base model and both runs: 40 images per model, 120 in all. Same
sampler for every model (DPM-Solver++, 30 steps, guidance 7.5). The default NSFW safety checker stays on; a flagged
image comes back black and is recorded in `metadata.csv`.

In [ ]:
from diffusers import DPMSolverMultistepScheduler
prompts = pd.read_csv("part2_diffusion/eval/prompts.csv")
pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16).to(dev)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)
meta_path = f"{OUT}/images/metadata.csv"
done = pd.read_csv(meta_path).to_dict("records") if os.path.exists(meta_path) else []
have = {(m["model"], m["prompt_id"], m["seed"]) for m in done}
t0, n_new = time.time(), 0
for model in MODELS:
    if model != "base":
        pipe.load_lora_weights(f"{OUT}/loras/{model}")
    os.makedirs(f"{OUT}/images/{model}", exist_ok=True)
    for r in prompts.itertuples():
        for seed in (0, 1):
            if (model, r.prompt_id, seed) in have:
                continue
            res = pipe(r.prompt, num_inference_steps=30, guidance_scale=7.5,
                       generator=torch.Generator(dev).manual_seed(seed))
            file = f"images/{model}/{r.prompt_id}_s{seed}.png"
            res.images[0].save(f"{OUT}/{file}")
            done.append({"file_name": file, "model": model, "prompt_id": r.prompt_id, "category": r.category,
                         "prompt": r.prompt, "seed": seed,
                         "nsfw_flagged": bool(res.nsfw_content_detected and res.nsfw_content_detected[0])})
            pd.DataFrame(done).to_csv(meta_path, index=False); n_new += 1
    if model != "base":
        pipe.unload_lora_weights()
gens = pd.DataFrame(done)
print(len(gens), "images,", int(gens.nsfw_flagged.sum()), "flagged by the NSFW filter:",
      gens[gens.nsfw_flagged][["model", "prompt_id", "seed"]].values.tolist())
if n_new:
    print(round((time.time() - t0) / n_new, 1), "s/image")
del pipe; torch.cuda.empty_cache()

In [ ]:
def load(file):
    return Image.open(f"{OUT}/{file}").convert("RGB")

grid_ids = ["ID01", "ID02", "ID03", "ID04", "OD01", "OD02", "CP01", "CP02"]
fig, axes = plt.subplots(len(grid_ids), 3, figsize=(9, 3.1 * len(grid_ids)))
for i, pid in enumerate(grid_ids):
    for j, model in enumerate(MODELS):
        g = gens[(gens.prompt_id == pid) & (gens.model == model) & (gens.seed == 0)].iloc[0]
        axes[i, j].imshow(load(g.file_name)); axes[i, j].axis("off")
        axes[i, j].set_title(f"{pid} {model}" + (" [NSFW]" if g.nsfw_flagged else ""), fontsize=8)
    axes[i, 0].text(-20, 256, prompts.set_index("prompt_id").loc[pid, "prompt"][:45], rotation=90,
                    va="center", ha="right", fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/grids.jpg", dpi=70); plt.show()

## 2.4 Evaluation
- **KID** (primary): in-domain generations (10 prompts x 2 seeds) against the 122 test images, Inception features,
  subset size 10. Noisy with only 20 generated images; lower is better.
- **CLIP score** (primary): 100 x cosine(prompt, image), CLIP ViT-B/32, over all 20 prompts and in-domain only.
- **Diversity** (secondary): LPIPS (AlexNet) between the seed-0 and seed-1 image of each prompt.
- **Efficiency**: GPU-hours and peak GPU memory per training run.
NSFW-flagged (black) images are left out of every score.

In [ ]:
from torchmetrics.image.kid import KernelInceptionDistance
from transformers import CLIPModel, CLIPProcessor
import lpips

ok = gens[~gens.nsfw_flagged]
def uint8(imgs):
    return torch.stack([torch.from_numpy(np.asarray(im)).permute(2, 0, 1) for im in imgs])

kid_scores = {}
real = uint8([im for im, _ in test])
for model in MODELS:
    kid = KernelInceptionDistance(subset_size=10).to(dev)
    for i in range(0, len(real), 32):
        kid.update(real[i:i + 32].to(dev), real=True)
    fake = uint8([load(f) for f in ok[(ok.model == model) & (ok.category == "in_domain")].file_name])
    kid.update(fake.to(dev), real=False)
    m, s = kid.compute(); kid_scores[model] = (m.item(), s.item())
    del kid; torch.cuda.empty_cache()

clip_model, clip_proc = CLIPModel.from_pretrained(CLIP).to(dev).eval(), CLIPProcessor.from_pretrained(CLIP)
def feats(out):
    return getattr(out, "pooler_output", out)        # transformers 5 returns an output object, 4.x a tensor

@torch.no_grad()
def image_emb(imgs):
    out = []
    for i in range(0, len(imgs), 32):
        x = clip_proc(images=imgs[i:i + 32], return_tensors="pt").to(dev)
        out.append(F.normalize(feats(clip_model.get_image_features(pixel_values=x.pixel_values)), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def clip_score(file, text):
    x = clip_proc(text=[text], return_tensors="pt", padding=True, truncation=True).to(dev)
    t = F.normalize(feats(clip_model.get_text_features(input_ids=x.input_ids, attention_mask=x.attention_mask)), dim=-1)
    return max(100 * (image_emb([load(file)]) @ t.T).item(), 0)

clip_df = ok.assign(clip_score=[clip_score(f, p) for f, p in zip(ok.file_name, ok.prompt)])

lp = lpips.LPIPS(net="alex").to(dev)
def lpips_t(file):
    return to_tensor(load(file)).unsqueeze(0).to(dev)
pairs = []
for (model, pid), g in ok.groupby(["model", "prompt_id"]):
    if len(g) == 2:
        with torch.no_grad():
            pairs.append({"model": model, "prompt_id": pid,
                          "lpips": lp(lpips_t(g.file_name.iloc[0]), lpips_t(g.file_name.iloc[1])).item()})
lpips_df = pd.DataFrame(pairs)

results = pd.DataFrame({m: {
    "kid_mean": round(kid_scores[m][0], 4), "kid_std": round(kid_scores[m][1], 4),
    "clip_score": round(clip_df[clip_df.model == m].clip_score.mean(), 2),
    "clip_in_domain": round(clip_df[(clip_df.model == m) & (clip_df.category == "in_domain")].clip_score.mean(), 2),
    "lpips_between_seeds": round(lpips_df[lpips_df.model == m].lpips.mean(), 3),
    "nsfw_flagged": int(gens[(gens.model == m)].nsfw_flagged.sum()),
    "gpu_hours": stats[m]["gpu_hours"] if m in stats else None,
    "peak_gpu_memory_gb": stats[m]["peak_gpu_memory_gb"] if m in stats else None} for m in MODELS}).T
print(results.to_string())
print(clip_df.pivot_table(index="category", columns="model", values="clip_score").round(2))

In [ ]:
# Final run: the run with the lower KID (in-domain style match); CLIP and the grids are checked below.
FINAL_RUN = min(RUNS, key=lambda r: kid_scores[r][0])
print("final run:", FINAL_RUN)

### Memorization check
For 10 in-domain images of the final run (seed 0), the most similar training image by CLIP image embedding (cosine).
The 3 closest pairs are shown. A similarity close to 1.0 would suggest the model reproduces a training image.

In [ ]:
train_emb = image_emb([im for im, _ in train])
mem_rows = []
for g in ok[(ok.model == FINAL_RUN) & (ok.category == "in_domain") & (ok.seed == 0)].itertuples():
    sims = (image_emb([load(g.file_name)]) @ train_emb.T)[0]
    j = int(sims.argmax())
    mem_rows.append({"prompt_id": g.prompt_id, "file_name": g.file_name, "train_index": j,
                     "train_caption": train[j][1], "similarity": round(sims[j].item(), 4)})
mem = pd.DataFrame(mem_rows).sort_values("similarity", ascending=False)
print(mem[["prompt_id", "similarity", "train_caption"]].to_string(index=False))
fig, axes = plt.subplots(3, 2, figsize=(6, 9))
for k, r in enumerate(mem.head(3).itertuples()):
    axes[k, 0].imshow(load(r.file_name)); axes[k, 0].set_title(f"{r.prompt_id} generated", fontsize=8)
    axes[k, 1].imshow(train[r.train_index][0]); axes[k, 1].set_title(f"closest training image, cos {r.similarity}", fontsize=8)
    for a in axes[k]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/memorization_top3.jpg", dpi=70); plt.show()

### Failure modes
The final run's 4 lowest-CLIP images (where it followed the prompt least), plus the out-of-domain and compositional
rows of the grid above, to pick two failure modes for the report.

In [ ]:
worst = clip_df[clip_df.model == FINAL_RUN].nsmallest(4, "clip_score")
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for a, r in zip(axes, worst.itertuples()):
    a.imshow(load(r.file_name)); a.axis("off")
    a.set_title(f"{r.prompt_id} s{r.seed} CLIP {r.clip_score:.1f}\n{r.prompt[:40]}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/failure_examples.jpg", dpi=70); plt.show()

## Results and save

In [ ]:
dest = "part2_diffusion/metrics/zoheb"
os.makedirs(dest, exist_ok=True)
results.to_csv(f"{dest}/results_table.csv")
train_table.to_csv(f"{dest}/training_runs.csv")
clip_df.to_csv(f"{dest}/clip_scores.csv", index=False)
lpips_df.to_csv(f"{dest}/lpips.csv", index=False)
mem.to_csv(f"{dest}/memorization.csv", index=False)
json.dump({"data_prep": prep, "final_run": FINAL_RUN}, open(f"{dest}/summary.json", "w"), indent=1)
import shutil
for f in ["grids.jpg", "memorization_top3.jpg", "failure_examples.jpg"]:
    shutil.copy(f"{OUT}/{f}", f"{dest}/{f}")
print(sorted(os.listdir(dest)))
if ON_COLAB:   # metrics + figures for the repo; LoRAs and the 120 images stay on Drive for Hugging Face
    shutil.make_archive("/content/p2_zoheb_metrics", "zip", ".", dest)
    from google.colab import files
    files.download("/content/p2_zoheb_metrics.zip")